# Boosting, and what the evidence for it actually says

MichAl Academy, lesson 2.6.

Run each cell with **Shift+Enter**.

A random forest grows many trees independently and averages them. Boosting grows
them one at a time, and each new tree is fitted to what the running total still
gets wrong. That one change is the whole method.

This notebook measures it honestly, which turns out to mean reporting that it
does not beat a random forest on datasets this size.

Two of the cells fit a few hundred models between them and take about a minute
each. That wait is part of the lesson: it is what tuning costs.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits, load_breast_cancer, load_wine, load_diabetes
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              HistGradientBoostingClassifier, RandomForestRegressor,
                              HistGradientBoostingRegressor)
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import (StratifiedKFold, KFold, cross_val_score,
                                     train_test_split, GridSearchCV)

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


## 1. A tree that can only ask one question

Set the depth to 1 and a tree becomes a stump: one question about one pixel, one
answer either way. Separating handwritten 3s from 8s, that is not nothing.


In [ ]:
digits = load_digits()
pair = np.isin(digits.target, [3, 8])
X, y = digits.data[pair], (digits.target[pair] == 8).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)
print(f"{len(y_train)} training images, {len(y_test)} held back")

stump = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_train, y_train)
print(f"one stump:                {stump.score(X_test, y_test):.3f}")
print(f"one unlimited tree:       {DecisionTreeClassifier(random_state=0).fit(X_train, y_train).score(X_test, y_test):.3f}")


Which is worth a pause: a single question about a single pixel lands within one
point of a tree that was allowed to ask as many as it liked. "Weak learner" is a
technical term, not an insult.


## 2. Eighty stumps in a row

Now fit stumps in sequence, each one working on what the previous ones still get
wrong. `staged_predict` replays the running total after every round, so you can
watch it happen instead of taking it on trust.


In [ ]:
boost = GradientBoostingClassifier(
    n_estimators=80, max_depth=1, learning_rate=0.1, random_state=0
).fit(X_train, y_train)

train_stages = list(boost.staged_predict(X_train))
test_stages = list(boost.staged_predict(X_test))

print(f"{'round':>6} {'wrong in training':>18} {'wrong held back':>16} {'accuracy':>9}")
for r in (1, 2, 3, 5, 10, 20, 40, 80):
    tr = int((train_stages[r - 1] != y_train).sum())
    te = int((test_stages[r - 1] != y_test).sum())
    print(f"{r:>6} {tr:>18} {te:>16} {1 - te / len(y_test):>9.3f}")


Two things in that table.

**Eighty one-question trees beat one unlimited tree**, 0.963 against 0.907. Each
stump is nearly useless alone and the sequence is not.

**Round 2 is worse than round 1** on held-back images, 15 errors against 11. The
training column falls almost every round; the held-back column wanders. Boosting
reliably reduces the error it can see, and the error you care about only follows
on average.


## 3. Now the honest comparison

The received wisdom is that gradient boosting is the thing to reach for on
tabular data. Test it against a random forest on every tabular set that ships
with scikit-learn.


In [ ]:
print(f"{'dataset':>16} {'rows':>6} {'forest':>8} {'boosting':>9}")
for name, load in [("breast cancer", load_breast_cancer), ("wine", load_wine)]:
    Xt, yt = load(return_X_y=True)
    forest = cross_val_score(RandomForestClassifier(n_estimators=200, random_state=0), Xt, yt, cv=CV).mean()
    gb = cross_val_score(HistGradientBoostingClassifier(random_state=0), Xt, yt, cv=CV).mean()
    print(f"{name:>16} {len(yt):>6} {forest:>8.3f} {gb:>9.3f}")

Xd, yd = load_diabetes(return_X_y=True)
KF = KFold(n_splits=5, shuffle=True, random_state=0)
forest = cross_val_score(RandomForestRegressor(n_estimators=200, random_state=0), Xd, yd, cv=KF).mean()
gb = cross_val_score(HistGradientBoostingRegressor(random_state=0), Xd, yd, cv=KF).mean()
print(f"{'diabetes (r2)':>16} {len(yd):>6} {forest:>8.3f} {gb:>9.3f}")


The forest wins on two of the three, out of the box.

Do not skip past that. On datasets of a few hundred rows, with default settings,
there is no case here for preferring boosting. Anybody telling you otherwise is
repeating something they read.


## 4. What boosting actually offers

Boosting has more dials than a forest, which is a liability at default settings
and an advantage once you turn them. Search over a few.


In [ ]:
search = GridSearchCV(
    HistGradientBoostingClassifier(random_state=0),
    {
        "learning_rate": [0.03, 0.1, 0.3],
        "max_leaf_nodes": [7, 15, 31],
        "max_iter": [100, 300],
    },
    cv=CV,
)
Xc, yc = load_breast_cancer(return_X_y=True)
search.fit(Xc, yc)

print(f"forest, out of the box:  {cross_val_score(RandomForestClassifier(n_estimators=200, random_state=0), Xc, yc, cv=CV).mean():.3f}")
print(f"boosting, out of the box: {cross_val_score(HistGradientBoostingClassifier(random_state=0), Xc, yc, cv=CV).mean():.3f}")
print(f"boosting, after a search: {search.best_score_:.3f}")
print(f"  best settings: {search.best_params_}")


Tuning moves boosting past the forest, by 0.7 of a point, after eighteen fits.

That is the real trade and it is not a dramatic one at this scale. A forest is
good immediately and has a low ceiling. Boosting is mediocre immediately, has a
higher ceiling, and charges you a search to reach it. On a Kaggle leaderboard
0.7 points decides the competition; in a system nobody has evaluated properly
yet, it is noise.


## 5. The two dials that matter, and why they fight

Learning rate is how much of each new tree's correction gets applied. Number of
rounds is how many trees. They trade off directly, and seeing that once saves a
lot of confused tuning.


In [ ]:
rounds = (10, 30, 100, 300)
print(f"{'rate':>6} " + " ".join(f"{n:>7}" for n in rounds))
for rate in (0.5, 0.2, 0.05, 0.01):
    scores = []
    for n in rounds:
        m = GradientBoostingClassifier(
            n_estimators=n, learning_rate=rate, max_depth=2, random_state=0
        )
        scores.append(cross_val_score(m, Xc, yc, cv=CV).mean())
    print(f"{rate:>6} " + " ".join(f"{s:>7.3f}" for s in scores))


Read the bottom-left corner. A learning rate of 0.01 with 10 rounds scores
0.627, and 62.7% of this dataset is benign, so that model is answering "benign"
for everything. Ten tiny steps got it precisely nowhere. The same rate with 300
rounds reaches 0.946.

So a small learning rate is not "more careful", it is *slower*, and it is only
better if you also pay for the rounds. The usual advice, lower the rate and
raise the rounds, is one instruction rather than two.


## 6. What happens if you just keep going

Boosting drives the training error towards zero by construction. Check whether
that ruins the held-back score.


In [ ]:
print(f"{'rounds':>7} {'train':>7} {'held back':>10}")
for n in (10, 80, 400, 1500):
    m = GradientBoostingClassifier(
        n_estimators=n, max_depth=1, learning_rate=0.1, random_state=0
    ).fit(X_train, y_train)
    print(f"{n:>7} {m.score(X_train, y_train):>7.3f} {m.score(X_test, y_test):>10.3f}")


Training error reaches zero at around 400 rounds and stays there. Held-back
accuracy climbs to 0.972 and then does not move, even at 1,500 rounds.

Which is a more careful answer than the warning you usually hear. With shallow
trees and a modest learning rate, extra rounds here were **wasted rather than
harmful**. They cost time and bought nothing. Boosting can overfit, and does so
readily with deep trees or a high learning rate, but "more rounds always
overfits" is not what this measurement says.


## 7. The libraries

Everything above used scikit-learn, because it is already installed and needs no
download. In production you will more often see three other names:

- **XGBoost**, the implementation that made the method famous through Kaggle
- **LightGBM**, faster on large data, and the origin of the histogram approach
  that `HistGradientBoostingClassifier` above is modelled on
- **CatBoost**, which handles categorical columns without the encoding work from
  lesson 2.2

The algorithm is the same idea in all of them and so is the tuning advice. If
you learn the two dials in part 5, you can drive any of them.

Which raises the obvious objection to part 2. That comparison used
scikit-learn's boosting, so maybe the forest only won because the real
implementations are better. Test it rather than argue about it.


In [ ]:
try:
    from xgboost import XGBClassifier, XGBRegressor
    from lightgbm import LGBMClassifier, LGBMRegressor
    HAVE_LIBS = True
except ImportError:
    HAVE_LIBS = False
    print("xgboost/lightgbm not installed here. Colab and Kaggle both ship")
    print("them; locally:  pip install xgboost lightgbm")

from sklearn.model_selection import RepeatedStratifiedKFold, RepeatedKFold


def compare(name, X, y, classify):
    """Defaults only, no tuning. Repeated 5-fold, 2 repeats, so ten fits each."""
    if classify:
        cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=0)
        scoring = "accuracy"
        models = {
            "random forest": RandomForestClassifier(random_state=0, n_jobs=1),
            "sklearn hist GB": HistGradientBoostingClassifier(random_state=0),
        }
        if HAVE_LIBS:
            models["XGBoost"] = XGBClassifier(random_state=0, verbosity=0, n_jobs=1)
            models["LightGBM"] = LGBMClassifier(random_state=0, verbose=-1, n_jobs=1)
    else:
        cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=0)
        scoring = "r2"
        models = {
            "random forest": RandomForestRegressor(random_state=0, n_jobs=1),
            "sklearn hist GB": HistGradientBoostingRegressor(random_state=0),
        }
        if HAVE_LIBS:
            models["XGBoost"] = XGBRegressor(random_state=0, verbosity=0, n_jobs=1)
            models["LightGBM"] = LGBMRegressor(random_state=0, verbose=-1, n_jobs=1)

    print("")
    print(f"{name}  ({scoring}, mean of 10 folds +/- sd)")
    for label, m in models.items():
        sc = cross_val_score(m, X, y, cv=cv, scoring=scoring)
        print(f"  {label:<16} {sc.mean():.4f}  +/- {sc.std():.4f}")


compare("wine", *load_wine(return_X_y=True), classify=True)
compare("breast cancer", *load_breast_cancer(return_X_y=True), classify=True)
compare("diabetes", *load_diabetes(return_X_y=True), classify=False)


The names change nothing. The forest still takes wine, where it ties
scikit-learn's boosting at 0.9775 and both of the famous libraries score lower,
and it still takes diabetes by a clear margin, 0.4280 against XGBoost's 0.3126.
Boosting takes breast cancer, 0.9728 to 0.9622.

So two of three to the forest, exactly as part 2 had it, and XGBoost at defaults
is the **worst** of the four on two of the three datasets.

Now read the `+/-` column, because it is the more honest answer. On wine the
spread is 0.017 and the gap between first and last is 0.017. On diabetes the
spread is 0.10 and the gap is 0.12. These datasets are a few hundred rows and
they cannot tell these models apart. That is the finding: not "the forest is
better", but "at this scale, and at defaults, the question does not have an
answer, so pick on speed, interpretability, or what your team already runs."


## What to take from this

| Claim | What we measured |
|---|---|
| Boosting beats a forest on tabular data | False at this scale. Forest won 2 of 3 at defaults |
| Boosting has a higher ceiling | True, and it cost a grid search for 0.7 of a point |
| Many weak learners beat one strong one | True. 80 stumps 0.963, one unlimited tree 0.907 |
| More rounds always overfits | Not here. Held-back accuracy plateaued rather than falling |

The claim that *is* well supported is a different one. Grinsztajn and colleagues
found that tree-based models, boosting and random forests together, remain
state-of-the-art on medium-sized tabular data of around ten thousand samples,
against neural networks. That is the finding worth carrying: **on a table, try
trees before you try anything deep.** It does not say which kind of tree
ensemble, and on a few hundred rows our own measurement says the difference is
not worth arguing about.
